<a href="https://colab.research.google.com/github/Vasquez-505/Inteligent-Systems---CourseWork/blob/main/Project/Training_and_inference_using_Google_Drive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SLEAP-NN Training — Drosophila FTIR Leg Tracking

Trains a **single-instance UNet** confmap model using **separate TRAIN and VAL**
`.pkg.slp` packages exported from the SLEAP GUI.

**Steps:** Install → Mount Drive → Dataset selection → Extract packages → MASTER CONFIG → Train → Results → Inference

## 1 — Install sleap-nn

In [ ]:
!pip uninstall -qqq -y opencv-python opencv-contrib-python
!pip install -qqq "sleap-nn[torch-cuda-128]"

import sleap_nn, torch
print("sleap-nn version :", sleap_nn.__version__)
print("PyTorch version  :", torch.__version__)
print("CUDA available   :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU              :", torch.cuda.get_device_name(0))

## 2 — Export training job packages from SLEAP GUI

Do this **on your local machine** before uploading to Drive.

### TRAIN set
1. Open `Drosophila_TRAIN_set_set<X>.slp` in SLEAP (replace `<X>` with your set letter)
2. Menu → **Predict → Run Training...**
3. Pipeline: **Single Instance**
4. Click **Export Training Job Package** (do NOT click Run)
5. Save as `Drosophila_TRAIN_set_set<X>.pkg.slp`

### VAL set
1. Open `Drosophila_VAL_set_set<X>.slp`
2. Same menu and pipeline
3. Click **Export Training Job Package**
4. Save as `Drosophila_VAL_set_set<X>.pkg.slp`

> The model config in the exported YAML does not matter — MASTER CONFIG overwrites everything.
> You are exporting purely to bundle labeled frames + images into a portable package.

### Google Drive structure expected
```
MyDrive/sleap/
├── Drosophila_TRAIN_set_set<X>.pkg.slp
└── Drosophila_VAL_set_set<X>.pkg.slp
```

## 3 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive/")

## 4 — Dataset selection

Set `VIDEO_SET` **once here**. Every subsequent cell derives package names,
channel count, and preprocessing flags from this single variable.

| Set | Folder | Channels | Appearance |
|-----|--------|----------|------------|
| `"A"` | `PROCESSED_Wrong_Res_NOVA_Videos/` | 3 (RGB) | Hot colourmap, NOVA resolution |
| `"B"` | `PROCESSED_Colormap_Videos/` | 3 (RGB) | Hot colourmap, original resolution |
| `"C"` | `PROCESSED_NO_Colormap_Videos/` | 1 (grey) | No colourmap, original resolution |
| `"D"` | `RAW_Videos/` | 1 (grey) | Raw FTIR, no processing |

Upload the corresponding `Drosophila_TRAIN_set_set<X>.pkg.slp` and
`Drosophila_VAL_set_set<X>.pkg.slp` to `MyDrive/sleap/` before running.

In [ ]:
# =====================================================================
# DATASET SELECTION  <-- THE ONLY CELL YOU NEED TO CHANGE BETWEEN RUNS
# =====================================================================
VIDEO_SET = "B"   # "A", "B", "C", or "D"
# =====================================================================

# Derived automatically -- do not edit below
IN_CHANNELS = 3 if VIDEO_SET in ("A", "B") else 1
ENSURE_RGB  = VIDEO_SET in ("A", "B")
ENSURE_GREY = not ENSURE_RGB

TRAIN_PKG = f"Drosophila_TRAIN_set_set{VIDEO_SET}.pkg.slp"
VAL_PKG   = f"Drosophila_VAL_set_set{VIDEO_SET}.pkg.slp"

print(f"Video set    : {VIDEO_SET}")
print(f"in_channels  : {IN_CHANNELS}")
print(f"ensure_rgb   : {ENSURE_RGB}")
print(f"TRAIN package: {TRAIN_PKG}")
print(f"VAL package  : {VAL_PKG}")

## 5 — Verify packages

`.pkg.slp` files are **HDF5 files** (not zip archives) — they already embed all labeled
frames and images internally. This cell verifies both files exist and sets `TRAIN_SLP` /
`VAL_SLP` for use by MASTER CONFIG.

In [ ]:
import os

SLEAP_DIR = "/content/drive/MyDrive/sleap"
os.chdir(SLEAP_DIR)
print("Working directory:", os.getcwd())
print("Files found:")
for f in sorted(os.listdir(".")): print(" ", f)

# TRAIN_PKG / VAL_PKG are set by the Dataset Selection cell above.
# .pkg.slp files are HDF5 — used directly as label paths, no extraction needed.
TRAIN_SLP = os.path.join(SLEAP_DIR, TRAIN_PKG)
VAL_SLP   = os.path.join(SLEAP_DIR, VAL_PKG)

for path in [TRAIN_SLP, VAL_SLP]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Not found: {path}")
    size_mb = os.path.getsize(path) / 1e6
    print(f"\nFound: {os.path.basename(path)}  ({size_mb:.1f} MB)")

print("\nTRAIN_SLP :", TRAIN_SLP)
print("VAL_SLP   :", VAL_SLP)

In [ ]:
import sleap_io as sio
import os

print("Patching video paths for Colab compatibility...")
print("(Replacing Windows paths → Colab Drive paths)\n")

for slp_path in [TRAIN_SLP, VAL_SLP]:
    pkg_name = os.path.basename(slp_path)
    print(f"{'='*60}")
    print(f"File   : {pkg_name}")

    labels = sio.load_slp(slp_path)
    print(f"Frames : {len(labels)}")
    print(f"Videos : {len(labels.videos)}")

    changed = 0
    for i, video in enumerate(labels.videos):
        fn = getattr(video, "filename", None) or ""
        is_windows = "Users" in fn or fn.lower().startswith("c:")
        if is_windows:
            basename = os.path.basename(fn)
            if pkg_name in basename:
                # Self-reference: embedded frames pointing to Windows .pkg.slp path
                new_fn = slp_path
            else:
                # External .avi reference: point to Drive (may not exist — OK if frames embedded)
                new_fn = os.path.join(SLEAP_DIR, basename)
            print(f"  video[{i}]: ...{fn[-40:]} → {new_fn}")
            video.filename = new_fn
            changed += 1

    if changed:
        print(f"Saving with {changed} patched path(s)...")
        sio.save_slp(labels, slp_path, embed=True)
        print(f"Saved  : {os.path.getsize(slp_path)/1e6:.1f} MB  ✓")
    else:
        print("No Windows paths found — already patched or paths are OK  ✓")

print(f"\n{'='*60}")
print("Path patching complete. Run Steps 6 → 7 to start training.")

## 5.5 — Patch video paths for Colab

`.pkg.slp` files exported from SLEAP GUI embed frames in HDF5 but store internal
video path references pointing to the original Windows machine (e.g. `C:\Users\pepev\...`).
sleap-nn validates these paths at startup and fails if they don't exist on Colab.

This cell patches those paths to the Colab Drive location of the `.pkg.slp` file itself,
so sleap-nn can find the embedded frame data. **Run once after uploading to Drive.**

## 6 — MASTER CONFIG

Builds `single_instance.yaml` from scratch with your chosen hyperparameters.

**Edit only the `TRAINING SETTINGS` block.** All other values come from the cells above.

| Variable | Source |
|----------|--------|
| `VIDEO_SET`, `IN_CHANNELS`, `ENSURE_RGB/GREY` | Dataset Selection cell (step 4) |
| `TRAIN_SLP`, `VAL_SLP` | Verify cell (step 5) |
| `BACKBONE`, `FILTERS`, `BATCH_SIZE`, `MAX_EPOCHS`, `USE_AUG` | This cell |

**Available backbones:**

| `BACKBONE` | Description | Notes |
|------------|-------------|-------|
| `"unet"` | Standard UNet encoder-decoder | SLEAP 2022 default for flies; fastest |
| `"convnext"` | ConvNeXt-v2 (modern CNN) | Better features, slower; try if UNet plateaus |
| `"swint"` | Swin Transformer | Vision transformer; highest capacity, slowest |

**Default parameters follow SLEAP 2022 (Pereira et al., Nature Methods):**
- UNet filters=64, output_stride=4, sigma=2.5 px (matches FTIR 1–3 px contact spots)
- Adam + AMSGrad, lr=1e-4; plateau scheduler factor=0.5, patience=20, min_lr=1e-6
- Early stopping patience=20
- Geometric augmentation only (no intensity — FTIR physics)

In [ ]:
import os, yaml, datetime, pathlib

# =====================================================================
# TRAINING SETTINGS  <-- EDIT THESE
# =====================================================================
BACKBONE       = "convnext" # "unet" | "convnext" | "swint"
FILTERS        = 64         # UNet only: base filters — 32 | 64
BATCH_SIZE     = 6          # reduce to 4 if GPU OOM
MAX_EPOCHS     = 200
USE_AUG        = True       # geometric augmentation: rotation ±15°, scale 0.9–1.1
USE_PRETRAINED = True       # ConvNeXt/SwinT only: True = ImageNet weights (forces RGB)
# =====================================================================

assert BACKBONE in ("unet", "convnext", "swint"), f"Unknown BACKBONE: {BACKBONE}"

# Torchvision ImageNet weights are 3-channel. If using pretrained ConvNeXt/SwinT,
# force RGB input even for grayscale source sets.
if USE_PRETRAINED and BACKBONE in ("convnext", "swint"):
    IN_CHANNELS = 3
    ENSURE_RGB = True
    ENSURE_GREY = False

_pt_tag  = "_pt" if (USE_PRETRAINED and BACKBONE != "unet") else ""
RUN_NAME = (f"drosophila_{BACKBONE}{FILTERS if BACKBONE == 'unet' else ''}{_pt_tag}_set{VIDEO_SET}_"
            + datetime.datetime.now().strftime("%y%m%d_%H%M%S"))
CKPT_DIR = os.path.join(SLEAP_DIR, "models")
pathlib.Path(CKPT_DIR).mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# Pretrained weight strings (torchvision ImageNet-1K V1)
# ------------------------------------------------------------------
_CONVNEXT_WEIGHTS = {
    "tiny" : "ConvNeXt_Tiny_Weights",
    "small": "ConvNeXt_Small_Weights",
    "base" : "ConvNeXt_Base_Weights",
    "large": "ConvNeXt_Large_Weights",
}
_SWINT_WEIGHTS = {
    "tiny" : "Swin_T_Weights",
    "small": "Swin_S_Weights",
    "base" : "Swin_B_Weights",
}

# ------------------------------------------------------------------
# Backbone configs — only the selected one is active; others are None
# ------------------------------------------------------------------
_unet_cfg = {
    # SLEAP 2022 default UNet for fly dataset — no pretrained weights available
    "in_channels"    : IN_CHANNELS,
    "kernel_size"    : 3,
    "filters"        : FILTERS,
    "filters_rate"   : 1.5,
    "max_stride"     : 32,
    "stem_stride"    : None,
    "middle_block"   : True,
    "up_interpolate" : True,   # bilinear upsampling — no checkerboard artefacts
    "stacks"         : 1,
    "convs_per_block": 2,
    "output_stride"  : 4,
} if BACKBONE == "unet" else None

_MODEL_TYPE_CONVNEXT = "tiny"   # "tiny" | "small" | "base" | "large"
_convnext_cfg = {
    # ConvNeXt — modern CNN backbone; optional ImageNet pretrained encoder
    "in_channels"       : IN_CHANNELS,
    "model_type"        : _MODEL_TYPE_CONVNEXT,
    "arch"              : None,
    "kernel_size"       : 3,
    "output_stride"     : 4,
    "up_interpolate"    : True,
    "pre_trained_weights": _CONVNEXT_WEIGHTS[_MODEL_TYPE_CONVNEXT] if USE_PRETRAINED else None,
} if BACKBONE == "convnext" else None

_MODEL_TYPE_SWINT = "tiny"      # "tiny" | "small" | "base"
_swint_cfg = {
    # Swin Transformer — vision transformer; optional ImageNet pretrained encoder
    "in_channels"       : IN_CHANNELS,
    "model_type"        : _MODEL_TYPE_SWINT,
    "arch"              : None,
    "patch_size"        : 4,
    "output_stride"     : 4,
    "up_interpolate"    : True,
    "pre_trained_weights": _SWINT_WEIGHTS[_MODEL_TYPE_SWINT] if USE_PRETRAINED else None,
} if BACKBONE == "swint" else None

# ------------------------------------------------------------------
# Full config
# ------------------------------------------------------------------
si = {
    "name"       : RUN_NAME,
    "description": "",
    "data_config": {
        "train_labels_path"  : [TRAIN_SLP],
        "val_labels_path"    : [VAL_SLP],
        "validation_fraction": 0,           # 0 = use dedicated VAL file
        "test_file_path"     : None,
        "provider"           : "LabelsReader",
        "user_instances_only": True,
        "data_pipeline_fw"   : "torch_dataset",
        "cache_img_path"     : None,
        "use_existing_imgs"  : False,
        "delete_cache_imgs_after_training": True,
        "preprocessing": {
            "ensure_rgb"      : ENSURE_RGB,
            "ensure_grayscale": ENSURE_GREY,
            "max_height"      : None,
            "max_width"       : None,
            "scale"           : 1.0,
            "crop_size"       : None,
            "min_crop_size"   : 100,
        },
        "use_augmentations_train": USE_AUG,
        "augmentation_config": {
            "intensity": {
                # ALL intensity augmentations OFF.
                # FTIR background is near-zero after subtraction.
                # Additive brightness creates spurious bright spots
                # indistinguishable from real leg-contact signals.
                "uniform_noise_p" : 0.0,
                "gaussian_noise_p": 0.0,
                "contrast_p"      : 0.0,
                "brightness_p"    : 0.0,
            },
            "geometric": {
                "rotation_min"    : -15.0,
                "rotation_max"    :  15.0,
                "scale_min"       :  0.9,
                "scale_max"       :  1.1,
                "translate_width" :  0.0,
                "translate_height":  0.0,
                "affine_p"        :  1.0 if USE_AUG else 0.0,
                "erase_p"         :  0.0,
                "mixup_p"         :  0.0,
            },
        },
        "skeletons": None,
    },
    "model_config": {
        "init_weights"               : "default",
        "pretrained_backbone_weights": None,
        "pretrained_head_weights"    : None,
        "backbone_config": {
            "unet"    : _unet_cfg,
            "convnext": _convnext_cfg,
            "swint"   : _swint_cfg,
        },
        "head_configs": {
            "single_instance": {
                "confmaps": {
                    "part_names"   : None,
                    # sigma=2.5 px matches FTIR contact spot size (1–3 px wide)
                    "sigma"        : 2.5,
                    "output_stride": 4,
                }
            },
            "centroid"            : None,
            "centered_instance"   : None,
            "bottomup"            : None,
            "multi_class_bottomup": None,
            "multi_class_topdown" : None,
        },
        "total_params": None,
    },
    "trainer_config": {
        "train_data_loader": {"batch_size": BATCH_SIZE, "shuffle": True,  "num_workers": 2},
        "val_data_loader"  : {"batch_size": BATCH_SIZE, "shuffle": False, "num_workers": 2},
        "model_ckpt"       : {"save_top_k": 1, "save_last": True},
        "trainer_devices"  : 1,
        "trainer_accelerator": "gpu",
        "enable_progress_bar": True,
        "train_steps_per_epoch": None,
        "max_epochs"       : MAX_EPOCHS,
        "seed"             : 42,
        "use_wandb"        : False,
        "save_ckpt"        : True,
        "ckpt_dir"         : CKPT_DIR,
        # ReduceLROnPlateau — reduce LR when val loss stops improving (SLEAP 2022)
        # factor=0.5: halve LR each time (PyTorch default 0.1 is too aggressive)
        # min_lr=1e-6: floor to prevent LR collapsing to zero
        "lr_scheduler": {
            "reduce_lr_on_plateau": {
                "threshold_mode": "rel",
                "factor"        : 0.5,
                "patience"      : 20,
                "min_lr"        : 1e-6,
            }
        },
        # Adam + AMSGrad — optimizer used in SLEAP 2022 (Pereira et al.)
        "optimizer_name": "Adam",
        "optimizer"     : {"lr": 1e-4, "amsgrad": True},
        "early_stopping": {
            "stop_training_on_plateau": True,
            "min_delta": 1e-8,
            "patience" : 20,
        },
        "run_name": RUN_NAME,
    },
}

OUT_YAML = os.path.join(SLEAP_DIR, "single_instance.yaml")
with open(OUT_YAML, "w") as f:
    yaml.safe_dump(si, f, sort_keys=False)

print(f"Config written : {OUT_YAML}")
print(f"Run name       : {RUN_NAME}")
print(f"Backbone       : {BACKBONE}")
if BACKBONE == "unet":
    print(f"filters        : {FILTERS}")
elif BACKBONE == "convnext":
    print(f"model_type     : {_MODEL_TYPE_CONVNEXT}")
    print(f"pretrained     : {USE_PRETRAINED}  ({'ImageNet-1K' if USE_PRETRAINED else 'scratch'})")
elif BACKBONE == "swint":
    print(f"model_type     : {_MODEL_TYPE_SWINT}")
    print(f"pretrained     : {USE_PRETRAINED}  ({'ImageNet-1K' if USE_PRETRAINED else 'scratch'})")
print(f"TRAIN labels   : {TRAIN_SLP}")
print(f"VAL labels     : {VAL_SLP}")
print(f"in_channels    : {IN_CHANNELS}  (video set {VIDEO_SET})")
print(f"batch_size     : {BATCH_SIZE}")
print(f"max_epochs     : {MAX_EPOCHS}")
print(f"augmentation   : {USE_AUG}  (geometry only, intensity OFF)")
print(f"optimizer      : Adam + AMSGrad, lr=1e-4  (SLEAP 2022)")
print(f"lr_scheduler   : ReduceLROnPlateau factor=0.5, patience=20, min_lr=1e-6")
print(f"sigma          : 2.5 px  (FTIR contact spot size)")


## 7 — Train

All paths are inside `single_instance.yaml` — no package argument needed.

In [ ]:
!python -m sleap_nn.cli train single_instance.yaml

## 8 — Training results

In [ ]:

# ============================================================
# STEP 8 — Comprehensive Training Results & Evaluation
# ============================================================
# Sources used:
#   training_log.csv          → loss curves, per-keypoint train loss, LR
#   metrics.{train,val}.0.npz → SLEAP built-in metrics (mOKS, mAP, distances, PCK, visibility)
#   labels_pr.{split}.0.slp   → predicted (x,y) per frame
#   labels_gt.{split}.0.slp   → ground-truth (x,y), NaN = airborne/invisible
# Custom metrics:
#   MPE   — mean position error (visible keypoints only)
#   FP/FN — false contact / missed contact rates
#   TD    — tracking drift (mean frame-to-frame displacement)
#   CR_kp — % of keypoint-frames needing correction (τ px threshold)
#   CR_frame — % of frames needing at least one correction
# ============================================================

import os, glob, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import sleap_io as sio

# ---------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------
TAU_PX    = 5.0    # correction threshold — errors > TAU_PX px count as a miss
SAVE_FIGS = True   # save figures to model directory as PNG

plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

KP_NAMES = ["head", "thorax", "abdomen",
            "forelegR", "forelegL", "midlegR", "midlegL", "hindlegR", "hindlegL"]
N_KP = len(KP_NAMES)

def kp_short(name):
    return {"forelegR":"fR","forelegL":"fL","midlegR":"mR",
            "midlegL":"mL","hindlegR":"hR","hindlegL":"hL"}.get(name, name[:6])

SHORT = [kp_short(k) for k in KP_NAMES]

# ---------------------------------------------------------------
# 0 · Find latest model directory
# ---------------------------------------------------------------
MODELS_DIR = os.path.join(SLEAP_DIR, "models")
runs = sorted(glob.glob(os.path.join(MODELS_DIR, "*")), key=os.path.getmtime)
if not runs:
    raise FileNotFoundError(f"No model directories found in {MODELS_DIR}")
MODEL_DIR = runs[-1]
nested = os.path.join(MODEL_DIR, os.path.basename(MODEL_DIR))
if os.path.isdir(nested):
    MODEL_DIR = nested
print(f"{'='*65}")
print(f"Model : {MODEL_DIR}")
print(f"{'='*65}")

# ---------------------------------------------------------------
# 1 · Training log → loss curves + LR + per-keypoint train loss
# ---------------------------------------------------------------
log_path = os.path.join(MODEL_DIR, "training_log.csv")
df = pd.read_csv(log_path)
# Normalise "/" → "_" in column names for easy access
df.columns = [c.replace("/", "_") for c in df.columns]
print(f"\nTraining log: {len(df)} epochs | columns: {list(df.columns)}")

# Auto-detect column names (flexible naming)
def find_col(df, *keywords_required, keywords_exclude=()):
    for c in df.columns:
        cl = c.lower()
        if all(k in cl for k in keywords_required) and not any(k in cl for k in keywords_exclude):
            return c
    return None

train_col = find_col(df, "train", "loss", keywords_exclude=("confmaps",)) or df.columns[1]
val_col   = find_col(df, "val",   "loss", keywords_exclude=("confmaps",)) or df.columns[2]
ep_col    = find_col(df, "epoch") or df.columns[0]
lr_col    = find_col(df, "learning_rate") or find_col(df, "lr")

best_idx   = df[val_col].idxmin()
best_epoch = int(df.loc[best_idx, ep_col])
best_val   = float(df.loc[best_idx, val_col])
best_train = float(df.loc[best_idx, train_col])
print(f"Best epoch: {best_epoch} | val_loss={best_val:.5f} | train_loss={best_train:.5f} | ratio={best_val/best_train:.3f}")

# ── Figure 1: loss curves + LR ──────────────────────────────────
ncols = 2 if lr_col else 1
fig1, axes1 = plt.subplots(1, ncols, figsize=(7*ncols, 4))
if ncols == 1:
    axes1 = [axes1]

ax = axes1[0]
ax.plot(df[ep_col], df[train_col], label="Train", lw=2, color="steelblue")
ax.plot(df[ep_col], df[val_col],   label="Val",   lw=2, color="tomato")
ax.axvline(best_epoch, color="gray", ls="--", alpha=0.7, label=f"Best={best_epoch}")
ax.set(xlabel="Epoch", ylabel="Loss", title="Loss Curves")
ax.legend(); ax.grid(True, alpha=0.3)

if lr_col:
    ax2 = axes1[1]
    ax2.semilogy(df[ep_col], df[lr_col], color="mediumseagreen", lw=2)
    ax2.axvline(best_epoch, color="gray", ls="--", alpha=0.7)
    ax2.set(xlabel="Epoch", ylabel="Learning Rate (log)", title="LR Schedule")
    ax2.grid(True, alpha=0.3)

plt.suptitle(os.path.basename(MODEL_DIR), fontsize=10, y=1.01)
plt.tight_layout()
if SAVE_FIGS:
    fig1.savefig(os.path.join(MODEL_DIR, "eval_loss_curves.png"), bbox_inches="tight")
plt.show()

# ── Figure 2: per-keypoint train loss at best epoch ─────────────
kp_cols = [c for c in df.columns if "confmaps" in c.lower()]
if kp_cols:
    kp_labels = [re.sub(r".*confmaps_?", "", c) for c in kp_cols]
    kp_vals   = df.loc[best_idx, kp_cols].values.astype(float)
    fig2, ax2 = plt.subplots(figsize=(11, 4))
    bars = ax2.bar(kp_labels, kp_vals, color="steelblue", alpha=0.8)
    ax2.bar_label(bars, fmt="%.4f", fontsize=8, padding=2)
    ax2.set(ylabel="Train Loss", title=f"Per-Keypoint Train Loss — Epoch {best_epoch}")
    ax2.tick_params(axis="x", rotation=30); ax2.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    if SAVE_FIGS:
        fig2.savefig(os.path.join(MODEL_DIR, "eval_kp_train_loss.png"), bbox_inches="tight")
    plt.show()
else:
    print("No per-keypoint loss columns found in training_log.csv")

# ---------------------------------------------------------------
# 2 · SLEAP built-in metrics from metrics.{split}.0.npz
# ---------------------------------------------------------------
def load_npz_metrics(npz_path):
    if not os.path.exists(npz_path):
        return None
    m = np.load(npz_path, allow_pickle=True)
    return m["metrics"].item() if "metrics" in m else dict(m)

def parse_npz(d):
    if d is None:
        return {}
    out = {}
    try: out["mOKS"]         = float(np.squeeze(d["mOKS"]))
    except: pass
    try:
        vm = d["visibility_metrics"]
        out.update({"vis_TP": int(vm["tp"]), "vis_FP": int(vm["fp"]),
                    "vis_TN": int(vm["tn"]), "vis_FN": int(vm["fn"]),
                    "vis_precision": float(vm["precision"]),
                    "vis_recall"   : float(vm["recall"])})
    except: pass
    try:
        dm = d["distance_metrics"]
        for k in ["avg","p50","p75","p90","p95","p99"]:
            if k in dm: out[f"dist_{k}"] = float(np.nanmean(dm[k]))
    except: pass
    try:
        pm = d["pck_metrics"]
        out["mPCK"]   = float(pm["mPCK"])
        out["PCK@5px"]= float(pm.get("PCK@5", pm.get("pck@5", float("nan"))))
        if "PCK@10" in pm: out["PCK@10px"] = float(pm["PCK@10"])
    except: pass
    try:
        vm2 = d["voc_metrics"]
        out["mAP"] = float(vm2["mAP"])
        out["mAR"] = float(vm2["mAR"])
    except: pass
    return out

print(f"\n{'─'*65}")
sleap_metrics = {}
for split in ("train", "val"):
    npz  = os.path.join(MODEL_DIR, f"metrics.{split}.0.npz")
    ms   = parse_npz(load_npz_metrics(npz))
    sleap_metrics[split] = ms
    if ms:
        print(f"\nSLEAP built-in metrics — {split.upper()}")
        for k, v in ms.items():
            print(f"  {k:<22} {v:.4f}" if isinstance(v, float) else f"  {k:<22} {v}")
    else:
        print(f"\n[metrics.{split}.0.npz not found]")

# ---------------------------------------------------------------
# 3 · Custom metrics — load labels_pr / labels_gt
# ---------------------------------------------------------------
def load_predictions(slp_path):
    """Load .slp → dict{frame_idx: (N_KP, 2)} with NaN for invisible."""
    if not os.path.exists(slp_path):
        print(f"  [Warning] Not found: {slp_path}")
        return None
    try:
        labels = sio.load_slp(slp_path)
    except Exception as e:
        print(f"  [Warning] Could not load {slp_path}: {e}")
        return None
    result = {}
    for lf in labels:
        if not lf.instances:
            continue
        inst = lf.instances[0]
        try:
            pts = inst.numpy()              # (N_KP, 2), NaN = invisible
        except Exception:
            try:
                pts = np.array([[p.x, p.y] for p in inst.points], dtype=float)
            except Exception:
                continue
        result[lf.frame_idx] = pts.astype(float)
    return result

def compute_custom(pts_pr, pts_gt, tau=5.0):
    """Compute per-keypoint custom metrics from prediction/GT dicts."""
    distances   = [[] for _ in range(N_KP)]
    tp = np.zeros(N_KP, int); fp = np.zeros(N_KP, int)
    tn = np.zeros(N_KP, int); fn = np.zeros(N_KP, int)
    n_vis = np.zeros(N_KP, int); n_invis = np.zeros(N_KP, int)

    frames      = sorted(pts_gt.keys())
    frame_error = {}

    for fidx in frames:
        if fidx not in pts_pr:
            continue
        gt = pts_gt[fidx]; pr = pts_pr[fidx]
        ferr = False
        for k in range(N_KP):
            gv = not (np.isnan(gt[k,0]) or np.isnan(gt[k,1]))
            pv = not (np.isnan(pr[k,0]) or np.isnan(pr[k,1]))
            if gv:
                n_vis[k] += 1
                if pv:
                    tp[k] += 1
                    d = float(np.hypot(pr[k,0]-gt[k,0], pr[k,1]-gt[k,1]))
                    distances[k].append(d)
                    if d > tau: ferr = True
                else:
                    fn[k] += 1; ferr = True
            else:
                n_invis[k] += 1
                if pv:
                    fp[k] += 1; ferr = True
                else:
                    tn[k] += 1
        frame_error[fidx] = ferr

    mpe     = np.array([np.nanmean(distances[k]) if distances[k] else np.nan for k in range(N_KP)])
    fp_rate = np.where(n_invis > 0, fp / n_invis, np.nan)
    fn_rate = np.where(n_vis   > 0, fn / n_vis,   np.nan)

    # Tracking Drift — mean |pred[t+1] − pred[t]| along visible run
    td = np.full(N_KP, np.nan)
    for k in range(N_KP):
        prev = None; disps = []
        for fidx in frames:
            if fidx not in pts_pr: continue
            pr = pts_pr[fidx]
            pv = not (np.isnan(pr[k,0]) or np.isnan(pr[k,1]))
            if pv:
                if prev is not None:
                    disps.append(float(np.hypot(pr[k,0]-prev[0], pr[k,1]-prev[1])))
                prev = pr[k]
            else:
                prev = None   # reset on gap
        if disps: td[k] = np.mean(disps)

    # CR_kp — errors per eligible keypoint-frame × 100
    err_counts = fp + fn + np.array([sum(1 for d in distances[k] if d > tau) for k in range(N_KP)])
    total_kpfr = n_vis + n_invis
    cr_kp = np.where(total_kpfr > 0, err_counts / total_kpfr * 100, np.nan)

    # CR_frame — % of frames with ≥1 error
    cr_frame = sum(frame_error.values()) / len(frame_error) * 100 if frame_error else np.nan

    # Global FP/FN (true ratios, not mean of per-kp rates)
    global_fp = fp.sum() / n_invis.sum() if n_invis.sum() > 0 else np.nan
    global_fn = fn.sum() / n_vis.sum()   if n_vis.sum()   > 0 else np.nan

    return dict(mpe=mpe, fp_rate=fp_rate, fn_rate=fn_rate,
                tp=tp, fp=fp, tn=tn, fn=fn,
                td=td, cr_kp=cr_kp, cr_frame=cr_frame,
                global_fp=global_fp, global_fn=global_fn,
                n_vis=n_vis, n_invis=n_invis,
                n_frames=len(frame_error))

custom_results = {}
for split in ("train", "val"):
    pr_path = os.path.join(MODEL_DIR, f"labels_pr.{split}.0.slp")
    gt_path = os.path.join(MODEL_DIR, f"labels_gt.{split}.0.slp")
    print(f"\n{'─'*65}")
    print(f"Custom metrics — {split.upper()}")
    pts_pr = load_predictions(pr_path)
    pts_gt = load_predictions(gt_path)
    if pts_pr is None or pts_gt is None:
        print("  [Skipped — files not found]")
        custom_results[split] = None
        continue
    print(f"  PR frames: {len(pts_pr)}  |  GT frames: {len(pts_gt)}")
    res = compute_custom(pts_pr, pts_gt, tau=TAU_PX)
    custom_results[split] = res
    total_vis   = int(res["n_vis"].sum())
    total_invis = int(res["n_invis"].sum())
    print(f"  Global MPE      : {np.nanmean(res['mpe']):.3f} px")
    print(f"  Global FP rate  : {res['global_fp']*100:.2f}%  "
          f"({int(res['fp'].sum())} FP / {total_invis} airborne kp-frames)")
    print(f"  Global FN rate  : {res['global_fn']*100:.2f}%  "
          f"({int(res['fn'].sum())} FN / {total_vis} contact kp-frames)")
    print(f"  Tracking Drift  : {np.nanmean(res['td']):.3f} px/frame  (mean across keypoints)")
    print(f"  CR_frame        : {res['cr_frame']:.1f}%  (frames needing ≥1 correction)")
    print(f"  CR_kp (mean)    : {np.nanmean(res['cr_kp']):.1f}%  (keypoint-frames needing correction)")

# ---------------------------------------------------------------
# 4 · Per-keypoint metric plots (one figure per split)
# ---------------------------------------------------------------
C = {"train": "steelblue", "val": "tomato"}

for split, res in custom_results.items():
    if res is None: continue

    fig, axes = plt.subplots(2, 3, figsize=(17, 8))
    fig.suptitle(f"{os.path.basename(MODEL_DIR)} — {split.upper()} · per-keypoint", fontsize=11)

    # MPE
    ax = axes[0,0]
    b = ax.bar(SHORT, res["mpe"], color=C[split], alpha=0.85)
    ax.bar_label(b, fmt="%.2f", fontsize=7, padding=2)
    ax.axhline(TAU_PX, color="red", ls="--", lw=1, alpha=0.6, label=f"τ={TAU_PX}px")
    ax.set(ylabel="px", title="Mean Position Error (MPE)")
    ax.legend(fontsize=8); ax.grid(True, axis="y", alpha=0.3)
    ax.tick_params(axis="x", rotation=30)

    # FP rate
    ax = axes[0,1]
    b = ax.bar(SHORT, res["fp_rate"]*100, color="salmon", alpha=0.85)
    ax.bar_label(b, fmt="%.1f%%", fontsize=7, padding=2)
    ax.set(ylabel="%", title="False Contact Rate  FP / airborne")
    ax.grid(True, axis="y", alpha=0.3); ax.tick_params(axis="x", rotation=30)

    # FN rate
    ax = axes[0,2]
    b = ax.bar(SHORT, res["fn_rate"]*100, color="gold", alpha=0.85)
    ax.bar_label(b, fmt="%.1f%%", fontsize=7, padding=2)
    ax.set(ylabel="%", title="Missed Contact Rate  FN / visible")
    ax.grid(True, axis="y", alpha=0.3); ax.tick_params(axis="x", rotation=30)

    # Tracking Drift
    ax = axes[1,0]
    b = ax.bar(SHORT, res["td"], color="mediumseagreen", alpha=0.85)
    ax.bar_label(b, fmt="%.2f", fontsize=7, padding=2)
    ax.set(ylabel="px/frame", title="Tracking Drift  mean frame-to-frame displacement")
    ax.grid(True, axis="y", alpha=0.3); ax.tick_params(axis="x", rotation=30)

    # CR_kp
    ax = axes[1,1]
    b = ax.bar(SHORT, res["cr_kp"], color="mediumpurple", alpha=0.85)
    ax.bar_label(b, fmt="%.1f%%", fontsize=7, padding=2)
    ax.set(ylabel="%", title=f"Correction Rate per Keypoint  (τ={TAU_PX}px)")
    ax.grid(True, axis="y", alpha=0.3); ax.tick_params(axis="x", rotation=30)

    # TP/FP/FN/TN counts
    ax = axes[1,2]
    x = np.arange(N_KP); w = 0.20
    ax.bar(x-1.5*w, res["tp"], w, label="TP", color="limegreen", alpha=0.8)
    ax.bar(x-0.5*w, res["fp"], w, label="FP", color="red",       alpha=0.7)
    ax.bar(x+0.5*w, res["fn"], w, label="FN", color="orange",    alpha=0.8)
    ax.bar(x+1.5*w, res["tn"], w, label="TN", color="silver",    alpha=0.7)
    ax.set_xticks(x); ax.set_xticklabels(SHORT, rotation=30, fontsize=8)
    ax.set(ylabel="count", title="Detection Counts per Keypoint")
    ax.legend(fontsize=8); ax.grid(True, axis="y", alpha=0.3)

    plt.tight_layout()
    if SAVE_FIGS:
        fig.savefig(os.path.join(MODEL_DIR, f"eval_kp_metrics_{split}.png"), bbox_inches="tight")
    plt.show()

# ---------------------------------------------------------------
# 5 · Combined summary table  (SLEAP + custom)
# ---------------------------------------------------------------
def fmt(v, decimals=4):
    try:    return f"{float(v):.{decimals}f}"
    except: return str(v)

summary_rows = []
for split in ("train", "val"):
    sm = sleap_metrics.get(split, {})
    cm = custom_results.get(split)
    row = {"split": split.upper()}
    if split == "val": row["best_epoch"] = best_epoch

    # SLEAP built-in
    for key, label in [("mOKS","mOKS"), ("mAP","mAP"), ("mAR","mAR"),
                       ("dist_avg","avg_dist_px"), ("dist_p50","p50_px"),
                       ("dist_p90","p90_px"),      ("dist_p95","p95_px"),
                       ("mPCK","mPCK"),            ("PCK@5px","PCK@5px"),
                       ("vis_precision","vis_Prec"),("vis_recall","vis_Rec"),
                       ("vis_FP","SLEAP_vis_FP"),  ("vis_FN","SLEAP_vis_FN")]:
        row[label] = fmt(sm.get(key, float("nan")))

    # Custom
    if cm is not None:
        row["MPE_px"]             = fmt(np.nanmean(cm["mpe"]), 3)
        row["FP_rate_%"]          = fmt(cm["global_fp"]*100,   2)
        row["FN_rate_%"]          = fmt(cm["global_fn"]*100,   2)
        row["TD_px_per_frame"]    = fmt(np.nanmean(cm["td"]),  3)
        row["CR_frame_%"]         = fmt(cm["cr_frame"],        1)
        row[f"CR_kp_tau{int(TAU_PX)}px_%"] = fmt(np.nanmean(cm["cr_kp"]), 1)
        row["n_frames"]           = cm["n_frames"]

    summary_rows.append(row)

df_sum = pd.DataFrame(summary_rows).set_index("split").T
print(f"\n{'='*65}")
print("SUMMARY TABLE")
print(f"{'='*65}")
with pd.option_context("display.max_rows", 60, "display.max_colwidth", 22):
    print(df_sum.to_string())

df_sum.to_csv(os.path.join(MODEL_DIR, "summary_metrics.csv"))
print(f"\nSaved → summary_metrics.csv")

# ---------------------------------------------------------------
# 6 · Per-keypoint detail table  (VAL)
# ---------------------------------------------------------------
res = custom_results.get("val")
if res is not None:
    df_kp = pd.DataFrame({
        "keypoint"   : KP_NAMES,
        "MPE_px"     : np.round(res["mpe"],        3),
        "FP_rate_%"  : np.round(res["fp_rate"]*100, 2),
        "FN_rate_%"  : np.round(res["fn_rate"]*100, 2),
        "TD_px/fr"   : np.round(res["td"],          3),
        "CR_kp_%"    : np.round(res["cr_kp"],        1),
        "TP": res["tp"], "FP": res["fp"], "FN": res["fn"], "TN": res["tn"],
        "n_contact"  : res["n_vis"],
        "n_airborne" : res["n_invis"],
    })
    print(f"\n{'='*65}")
    print("PER-KEYPOINT DETAIL — VAL")
    print(f"{'='*65}")
    print(df_kp.to_string(index=False))
    df_kp.to_csv(os.path.join(MODEL_DIR, "per_keypoint_metrics_val.csv"), index=False)
    print(f"\nSaved → per_keypoint_metrics_val.csv")

print(f"\n{'='*65}")
print(f"All outputs → {MODEL_DIR}")
print(f"{'='*65}")


## 8.5 — Optional: Evaluate on held-out TEST set

Run this cell only **after Step 8 has completed** — all helper functions and variables (`MODEL_DIR`, `TAU_PX`, `SAVE_FIGS`, `KP_NAMES`, `SHORT`, `load_predictions`, `compute_custom`, `parse_npz`, `fmt`, …) are inherited from that cell.

Produces the same outputs as Step 8 for the TEST split: SLEAP built-in metrics (mOKS, mAP, distances, PCK, visibility), custom thesis metrics (MPE, FP/FN rates, Tracking Drift, CR\_frame), per-keypoint figure, and CSVs.

**Requirement:** `Drosophila_TEST_set_set{VIDEO_SET}.pkg.slp` must be present in `MyDrive/sleap/` (patch its video paths with Step 5.5 logic if needed).

In [ ]:
# ============================================================
# STEP 8.5 — Optional: Evaluate on held-out TEST set
# ============================================================
# Inherits from Step 8: MODEL_DIR, TAU_PX, SAVE_FIGS, KP_NAMES,
# N_KP, SHORT, SLEAP_DIR, VIDEO_SET, load_npz_metrics, parse_npz,
# load_predictions, compute_custom, fmt, pd, np, plt, sio, os
# ============================================================

import subprocess, shutil

# ── TEST dataset path (same convention as TRAIN_PKG / VAL_PKG) ──
TEST_PKG = f"Drosophila_TEST_set_set{VIDEO_SET}.pkg.slp"
TEST_SLP = os.path.join(SLEAP_DIR, TEST_PKG)

if not os.path.exists(TEST_SLP):
    print(f"TEST dataset not found: {TEST_SLP}")
    print(f"Upload {TEST_PKG} to MyDrive/sleap/ and re-run this cell.")
else:
    pr_path  = os.path.join(MODEL_DIR, "labels_pr.test.0.slp")
    gt_path  = os.path.join(MODEL_DIR, "labels_gt.test.0.slp")
    npz_path = os.path.join(MODEL_DIR, "metrics.test.0.npz")

    print(f"{'='*65}")
    print(f"TEST EVALUATION — {os.path.basename(MODEL_DIR)}")
    print(f"TEST dataset     : {os.path.basename(TEST_SLP)}")
    print(f"{'='*65}")

    # ── 1. Copy GT labels into model folder ─────────────────────
    if not os.path.exists(gt_path):
        shutil.copy(TEST_SLP, gt_path)
        print(f"GT labels copied → {os.path.basename(gt_path)}")

    # ── 2. Run inference if predictions don't exist ─────────────
    if not os.path.exists(pr_path):
        print("\nRunning inference on TEST set …")
        r = subprocess.run(
            ["python", "-m", "sleap_nn.cli", "track",
             "--model_paths", MODEL_DIR,  # model DIRECTORY, not the yaml file
             "--data_path",   gt_path,
             "--output_path", pr_path],
            capture_output=True, text=True,
        )
        if r.returncode != 0:
            print("  Inference failed:\n", r.stderr[-2000:])
        else:
            print(f"  Predictions saved → {os.path.basename(pr_path)}")
    else:
        print(f"\nPredictions already exist: {os.path.basename(pr_path)}")

    # ── 3. SLEAP built-in eval (best-effort; may warn on empty pairs) ──
    if os.path.exists(pr_path) and not os.path.exists(npz_path):
        print("\nComputing SLEAP built-in metrics for TEST …")
        r = subprocess.run(
            ["python", "-m", "sleap_nn.cli", "eval",
             "--ground_truth_path", gt_path,
             "--predicted_path",    pr_path,
             "--save_metrics",      npz_path],
            capture_output=True, text=True,
        )
        if r.returncode != 0:
            print("  SLEAP eval warning:", r.stderr[-400:])
        elif os.path.exists(npz_path):
            print(f"  SLEAP metrics saved → metrics.test.0.npz")

    # ── 4. Display SLEAP built-in metrics ───────────────────────
    print(f"\n{'─'*65}")
    sm_test = parse_npz(load_npz_metrics(npz_path))
    if sm_test:
        print("SLEAP built-in metrics — TEST")
        for k, v in sm_test.items():
            print(f"  {k:<22} {v:.4f}" if isinstance(v, float) else f"  {k:<22} {v}")
    else:
        print("[SLEAP built-in metrics not available for TEST]")

    # ── 5. Custom metrics ────────────────────────────────────────
    pts_pr_t = load_predictions(pr_path)
    pts_gt_t = load_predictions(gt_path)
    res_t    = None

    if pts_pr_t and pts_gt_t:
        print(f"\n{'─'*65}")
        print(f"Custom metrics — TEST")
        print(f"  PR frames: {len(pts_pr_t)}  |  GT frames: {len(pts_gt_t)}")
        res_t = compute_custom(pts_pr_t, pts_gt_t, tau=TAU_PX)
        print(f"  Global MPE      : {np.nanmean(res_t['mpe']):.3f} px")
        print(f"  Global FP rate  : {res_t['global_fp']*100:.2f}%  "
              f"({int(res_t['fp'].sum())} FP / {int(res_t['n_invis'].sum())} airborne kp-frames)")
        print(f"  Global FN rate  : {res_t['global_fn']*100:.2f}%  "
              f"({int(res_t['fn'].sum())} FN / {int(res_t['n_vis'].sum())} contact kp-frames)")
        print(f"  Tracking Drift  : {np.nanmean(res_t['td']):.3f} px/frame")
        print(f"  CR_frame        : {res_t['cr_frame']:.1f}%  (frames needing ≥1 correction)")
        print(f"  CR_kp (mean)    : {np.nanmean(res_t['cr_kp']):.1f}%")

        # ── 6. Per-keypoint plot ─────────────────────────────────
        fig, axes = plt.subplots(2, 3, figsize=(17, 8))
        fig.suptitle(f"{os.path.basename(MODEL_DIR)} — TEST · per-keypoint", fontsize=11)

        ax = axes[0,0]
        b = ax.bar(SHORT, res_t["mpe"], color="darkorange", alpha=0.85)
        ax.bar_label(b, fmt="%.2f", fontsize=7, padding=2)
        ax.axhline(TAU_PX, color="red", ls="--", lw=1, alpha=0.6, label=f"τ={TAU_PX}px")
        ax.set(ylabel="px", title="Mean Position Error (MPE)")
        ax.legend(fontsize=8); ax.grid(True, axis="y", alpha=0.3)
        ax.tick_params(axis="x", rotation=30)

        ax = axes[0,1]
        b = ax.bar(SHORT, res_t["fp_rate"]*100, color="salmon", alpha=0.85)
        ax.bar_label(b, fmt="%.1f%%", fontsize=7, padding=2)
        ax.set(ylabel="%", title="False Contact Rate  FP / airborne")
        ax.grid(True, axis="y", alpha=0.3); ax.tick_params(axis="x", rotation=30)

        ax = axes[0,2]
        b = ax.bar(SHORT, res_t["fn_rate"]*100, color="gold", alpha=0.85)
        ax.bar_label(b, fmt="%.1f%%", fontsize=7, padding=2)
        ax.set(ylabel="%", title="Missed Contact Rate  FN / visible")
        ax.grid(True, axis="y", alpha=0.3); ax.tick_params(axis="x", rotation=30)

        ax = axes[1,0]
        b = ax.bar(SHORT, res_t["td"], color="mediumseagreen", alpha=0.85)
        ax.bar_label(b, fmt="%.2f", fontsize=7, padding=2)
        ax.set(ylabel="px/frame", title="Tracking Drift  mean frame-to-frame displacement")
        ax.grid(True, axis="y", alpha=0.3); ax.tick_params(axis="x", rotation=30)

        ax = axes[1,1]
        b = ax.bar(SHORT, res_t["cr_kp"], color="mediumpurple", alpha=0.85)
        ax.bar_label(b, fmt="%.1f%%", fontsize=7, padding=2)
        ax.set(ylabel="%", title=f"Correction Rate per Keypoint  (τ={TAU_PX}px)")
        ax.grid(True, axis="y", alpha=0.3); ax.tick_params(axis="x", rotation=30)

        ax = axes[1,2]
        x = np.arange(N_KP); w = 0.20
        ax.bar(x-1.5*w, res_t["tp"], w, label="TP", color="limegreen", alpha=0.8)
        ax.bar(x-0.5*w, res_t["fp"], w, label="FP", color="red",       alpha=0.7)
        ax.bar(x+0.5*w, res_t["fn"], w, label="FN", color="orange",    alpha=0.8)
        ax.bar(x+1.5*w, res_t["tn"], w, label="TN", color="silver",    alpha=0.7)
        ax.set_xticks(x); ax.set_xticklabels(SHORT, rotation=30, fontsize=8)
        ax.set(ylabel="count", title="Detection Counts per Keypoint")
        ax.legend(fontsize=8); ax.grid(True, axis="y", alpha=0.3)

        plt.tight_layout()
        if SAVE_FIGS:
            fig.savefig(os.path.join(MODEL_DIR, "eval_kp_metrics_test.png"), bbox_inches="tight")
        plt.show()

        # ── 7. Per-keypoint detail table ─────────────────────────
        df_kp_t = pd.DataFrame({
            "keypoint"  : KP_NAMES,
            "MPE_px"    : np.round(res_t["mpe"],           3),
            "FP_rate_%" : np.round(res_t["fp_rate"] * 100, 2),
            "FN_rate_%" : np.round(res_t["fn_rate"] * 100, 2),
            "TD_px/fr"  : np.round(res_t["td"],            3),
            "CR_kp_%"   : np.round(res_t["cr_kp"],          1),
            "TP": res_t["tp"], "FP": res_t["fp"],
            "FN": res_t["fn"], "TN": res_t["tn"],
            "n_contact" : res_t["n_vis"],
            "n_airborne": res_t["n_invis"],
        })
        print(f"\n{'='*65}")
        print("PER-KEYPOINT DETAIL — TEST")
        print(f"{'='*65}")
        print(df_kp_t.to_string(index=False))
        df_kp_t.to_csv(os.path.join(MODEL_DIR, "per_keypoint_metrics_test.csv"), index=False)
        print(f"\nSaved → per_keypoint_metrics_test.csv")

        # ── 8. Append TEST column to summary_metrics.csv ─────────
        row_t = {"split": "TEST"}
        for key, label in [("mOKS","mOKS"), ("mAP","mAP"), ("mAR","mAR"),
                            ("dist_avg","avg_dist_px"), ("dist_p50","p50_px"),
                            ("dist_p90","p90_px"),      ("dist_p95","p95_px"),
                            ("mPCK","mPCK"),            ("PCK@5px","PCK@5px"),
                            ("vis_precision","vis_Prec"),("vis_recall","vis_Rec"),
                            ("vis_FP","SLEAP_vis_FP"),  ("vis_FN","SLEAP_vis_FN")]:
            row_t[label] = fmt(sm_test.get(key, float("nan"))) if sm_test else float("nan")
        row_t["MPE_px"]          = fmt(np.nanmean(res_t["mpe"]), 3)
        row_t["FP_rate_%"]       = fmt(res_t["global_fp"] * 100, 2)
        row_t["FN_rate_%"]       = fmt(res_t["global_fn"] * 100, 2)
        row_t["TD_px_per_frame"] = fmt(np.nanmean(res_t["td"]),  3)
        row_t["CR_frame_%"]      = fmt(res_t["cr_frame"],        1)
        row_t[f"CR_kp_tau{int(TAU_PX)}px_%"] = fmt(np.nanmean(res_t["cr_kp"]), 1)
        row_t["n_frames"]        = res_t["n_frames"]

        summary_csv = os.path.join(MODEL_DIR, "summary_metrics.csv")
        if os.path.exists(summary_csv):
            df_s = pd.read_csv(summary_csv, index_col=0)
            df_s["TEST"] = pd.Series({k: v for k, v in row_t.items() if k != "split"})
            df_s.to_csv(summary_csv)
        else:
            pd.DataFrame([row_t]).set_index("split").T.to_csv(summary_csv)
        print(f"Summary updated  → summary_metrics.csv")
    else:
        print("[Custom metrics skipped — prediction or GT files not found]")

    print(f"\n{'='*65}")
    print(f"All TEST outputs → {MODEL_DIR}")
    print(f"{'='*65}")


## 9 — Inference on test video

Upload a test video to `MyDrive/sleap/` and edit `VIDEO_FILE` below.
Leave `MODEL_NAME` empty to auto-detect the latest model.

---
## ▲ TRAINING PIPELINE — Steps 1–8
**Run all above** (`Runtime → Run all` or right-click Step 9 → *Run all above*) to install, mount, configure, train, and evaluate in one go.

## ▼ INFERENCE — Steps 9–10
Run **manually and separately** after reviewing Step 8 results.
Requires a test video uploaded to `MyDrive/sleap/`.

---

In [ ]:
%%bash
# -- EDIT THESE ------------------------------------------------------
VIDEO_FILE="fly1.2.avi"
MODEL_NAME=""
# --------------------------------------------------------------------
BASE_DIR="/content/drive/MyDrive/sleap"
MODELS_DIR="${BASE_DIR}/models"
if [ -z "${MODEL_NAME}" ]; then
    MODEL_DIR=$(ls -dt "${MODELS_DIR}"/*/ 2>/dev/null | head -1)
    MODEL_DIR=${MODEL_DIR%/}
    echo "Auto-detected model: ${MODEL_DIR}"
else
    MODEL_DIR="${MODELS_DIR}/${MODEL_NAME}"
fi
PRED_DIR="${MODEL_DIR}/predictions"; mkdir -p "${PRED_DIR}"
BASENAME=$(basename "${VIDEO_FILE%.*}")
PRED_FILE="${PRED_DIR}/${BASENAME}.predictions.slp"
echo "Model       : ${MODEL_DIR}"
echo "Video       : ${BASE_DIR}/${VIDEO_FILE}"
echo "Predictions : ${PRED_FILE}"
sleap-track "${BASE_DIR}/${VIDEO_FILE}" --model "${MODEL_DIR}" --output "${PRED_FILE}"

## 10 — Render predictions video